<a href="https://colab.research.google.com/github/CollisionSeverityPrediction/T11/blob/main/Emma_Sloan_Ass1_T11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# importing Python libraries and functions

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [ ]:
# Loading the dataset into a pandas dataframe
df = pd.read_csv('/content/drive/MyDrive/Assignment_1/dft-road-casualty-statistics-collision-2024.csv')

/tmp/ipykernel_3021/2282544302.py:2: DtypeWarning: Columns (0,2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/Assignment_1/dft-road-casualty-statistics-collision-2024.csv')


In [ ]:
# drop irrelevant columns such as collision, district and road IDs as advised in brief
df.drop(columns=['collision_index', 'collision_year', 'collision_ref_no',
                  'location_easting_osgr', 'location_northing_osgr',
                  'police_force', 'local_authority_district', 'local_authority_ons_district',
                  'local_authority_highway', 'local_authority_highway_current',
                  'first_road_number', 'junction_detail_historic',
                  'pedestrian_crossing_human_control_historic',
                  'pedestrian_crossing_physical_facilities_historic',
                  'carriageway_hazards_historic', 'second_road_number',
                  'did_police_officer_attend_scene_of_accident',
                  'lsoa_of_accident_location'], inplace=True)

In [ ]:
df.head()

,longitude,latitude,collision_severity,number_of_vehicles,number_of_casualties,date,day_of_week,time,first_road_class,road_type,...,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards,urban_or_rural_area,trunk_road_flag,enhanced_severity_collision,collision_injury_based,collision_adjusted_severity_serious,collision_adjusted_severity_slight
0,-1.22722,54.57219,3,2,1,05/12/2024,5,16:10,6,6,...,2,2,-1,0,1,2,3,1,0.000000,1.000000
1,-1.29265,54.54466,3,1,1,22/10/2024,3,14:56,6,6,...,1,1,-1,0,1,2,3,1,0.000000,1.000000
2,-2.53157,53.41443,3,2,1,24/01/2024,4,17:50,6,6,...,1,1,0,0,2,2,-1,0,0.014743,0.985257
3,-2.51630,53.67359,2,1,1,22/05/2024,4,17:45,3,6,...,5,5,-1,0,2,2,6,1,1.000000,0.000000
4,-2.50416,53.76592,2,2,1,11/08/2024,1,15:36,3,3,...,1,1,-1,0,1,2,7,1,1.000000,0.000000


In [ ]:
# save target before dropping column
y = df['collision_severity']

In [ ]:
# drop additional columns - justified in report
df.drop(columns=['longitude', 'latitude', 'collision_severity',
                  'number_of_casualties', 'enhanced_severity_collision',
                  'collision_injury_based', 'collision_adjusted_severity_serious',
                  'collision_adjusted_severity_slight'], inplace=True)

In [ ]:
df.head()

,number_of_vehicles,date,day_of_week,time,first_road_class,road_type,speed_limit,junction_detail,junction_control,second_road_class,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,special_conditions_at_site,carriageway_hazards,urban_or_rural_area,trunk_road_flag
0,2,05/12/2024,5,16:10,6,6,30,0,-1,-1,0,4,2,2,-1,0,1,2
1,1,22/10/2024,3,14:56,6,6,30,0,-1,-1,0,1,1,1,-1,0,1,2
2,2,24/01/2024,4,17:50,6,6,30,13,4,6,0,4,1,1,0,0,2,2
3,1,22/05/2024,4,17:45,3,6,50,0,-1,-1,0,1,5,5,-1,0,2,2
4,2,11/08/2024,1,15:36,3,3,50,16,4,6,0,1,1,1,-1,0,1,2


In [ ]:
# get value counts
for col in ['first_road_class', 'road_type', 'speed_limit', 'junction_detail',
            'junction_control', 'second_road_class', 'pedestrian_crossing',
            'light_conditions', 'weather_conditions', 'road_surface_conditions',
            'special_conditions_at_site', 'carriageway_hazards',
            'urban_or_rural_area', 'trunk_road_flag', 'day_of_week']:
    print(col, df[col].value_counts(dropna=False).sort_index())

first_road_class first_road_class
1     3029
2      229
3    44142
4    12677
5     4384
6    36466
Name: count, dtype: int64
road_type road_type
1     6204
2     2224
3    14671
6    73618
7     1642
9     2568
Name: count, dtype: int64
speed_limit speed_limit
-1         3
 20    18881
 30    50257
 40     8949
 50     4678
 60    12709
 70     5450
Name: count, dtype: int64
junction_detail junction_detail
-1      7005
 0     46139
 13    28004
 16     9495
 17     1669
 18     2256
 19     6357
 99        2
Name: count, dtype: int64
junction_control junction_control
-1    42581
 1      418
 2    11027
 3      661
 4    44332
 9     1908
Name: count, dtype: int64
second_road_class second_road_class
-1    11167
 0    12848
 1      747
 2       76
 3    11899
 4     4478
 5     2773
 6    56939
Name: count, dtype: int64
pedestrian_crossing pedestrian_crossing
-1      1034
 0     74788
 11      377
 12     1001
 13     4115
 14     6273
 15     7342
 16      241
 17     2744
 99     3012

In [ ]:
# check for missing/unknown codes (-1, 9, 99) and null entries

summary = pd.DataFrame({
    'nan_count': df.isna().sum(),
    'nan_pct': (df.isna().sum() / len(df) * 100).round(2),
    'code_-1_count': (df == -1).sum(),
    'code_-1_pct': ((df == -1).sum() / len(df) * 100).round(2),
    'code_9_count': (df == 9).sum(),
    'code_9_pct': ((df == 9).sum() / len(df) * 100).round(2),
    'code_99_count': (df == 99).sum(),
    'code_99_pct': ((df == 99).sum() / len(df) * 100).round(2),
})

summary['total_missing_count'] = summary[['nan_count', 'code_-1_count', 'code_9_count', 'code_99_count']].sum(axis=1)
summary['total_missing_pct'] = summary[['nan_pct', 'code_-1_pct', 'code_9_pct', 'code_99_pct']].sum(axis=1)

summary = summary.sort_values('total_missing_pct', ascending=False)
summary

,nan_count,nan_pct,code_-1_count,code_-1_pct,code_9_count,code_9_pct,code_99_count,code_99_pct,total_missing_count,total_missing_pct
special_conditions_at_site,0,0.0,59299,58.75,2672,2.65,0,0.00,61971,61.40
junction_control,0,0.0,42581,42.19,1908,1.89,0,0.00,44489,44.08
second_road_class,0,0.0,11167,11.06,0,0.00,0,0.00,11167,11.06
trunk_road_flag,0,0.0,7080,7.01,0,0.00,0,0.00,7080,7.01
junction_detail,0,0.0,7005,6.94,0,0.00,2,0.00,7007,6.94
pedestrian_crossing,0,0.0,1034,1.02,0,0.00,3012,2.98,4046,4.00
weather_conditions,0,0.0,0,0.00,3145,3.12,0,0.00,3145,3.12
carriageway_hazards,0,0.0,421,0.42,0,0.00,2662,2.64,3083,3.06
road_type,0,0.0,0,0.00,2568,2.54,0,0.00,2568,2.54
road_surface_conditions,0,0.0,704,0.70,1670,1.65,0,0.00,2374,2.35


In [ ]:
# Does 'unknown/missing' rate differ across severity classes? Checking for features with >10% unknown/missing
for col in ['special_conditions_at_site', 'junction_control', 'second_road_class', 'trunk_road_flag', 'junction_detail']:
    is_missing = df[col].isin([-1, 9, 99])
    print(col)
    print(pd.crosstab(is_missing, y, normalize='index').round(3))
    print()

special_conditions_at_site
collision_severity              1      2      3
special_conditions_at_site                     
False                       0.012  0.195  0.794
True                        0.017  0.258  0.725

junction_control
collision_severity      1      2      3
junction_control                       
False               0.009  0.223  0.768
True                0.022  0.247  0.731

second_road_class
collision_severity      1      2      3
second_road_class                      
False               0.013  0.229  0.759
True                0.032  0.273  0.695

trunk_road_flag
collision_severity      1      2      3
trunk_road_flag                        
False               0.014  0.224  0.763
True                0.032  0.361  0.607

junction_detail
collision_severity      1      2      3
junction_detail                        
False               0.016  0.239  0.745
True                0.006  0.159  0.836



In [ ]:
# drop columns with high percentage of missing/unknown values that introduce data leakage
df.drop(columns=['special_conditions_at_site', 'junction_control', 'second_road_class', 'trunk_road_flag'], inplace=True)

In [ ]:
# Number of missing/unknown values per row
missing_per_row = (df.isna() | df.isin([-1, 9, 99])).sum(axis=1)
missing_per_row.value_counts().sort_index()

,count
0,88079
1,8765
2,1582
3,964
4,605
5,598
6,334


In [ ]:
# feature engineering for date and time. date becomes month and time becomes time_of_day
df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
df['time'] = pd.to_datetime(df['time'], format='%H:%M')

# Extract month
df['month'] = df['date'].dt.month

# Extract hour, then bucket it into time-of-day categories
df['hour'] = df['time'].dt.hour

def bucket_hour(hour):
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 10:
        return 'morning_rush'
    elif 10 <= hour < 16:
        return 'daytime'
    elif 16 <= hour < 19:
        return 'evening_rush'
    else:
        return 'evening'

df['time_of_day'] = df['hour'].apply(bucket_hour)

# Drop the now-redundant raw columns
df.drop(columns=['date', 'time', 'hour'], inplace=True)
X = df

In [ ]:
df.head()

,number_of_vehicles,day_of_week,first_road_class,road_type,speed_limit,junction_detail,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,carriageway_hazards,urban_or_rural_area,month,time_of_day
0,2,5,6,6,30,0,0,4,2,2,0,1,12,evening_rush
1,1,3,6,6,30,0,0,1,1,1,0,1,10,daytime
2,2,4,6,6,30,13,0,4,1,1,0,2,1,evening_rush
3,1,4,3,6,50,0,0,1,5,5,0,2,5,evening_rush
4,2,1,3,3,50,16,0,1,1,1,0,1,8,daytime


In [ ]:
missing_codes = [-1, 9, 99]
cols_needing_replace = ['junction_detail', 'pedestrian_crossing', 'weather_conditions',
                         'carriageway_hazards', 'road_type', 'road_surface_conditions',
                         'light_conditions', 'speed_limit']
X[cols_needing_replace] = X[cols_needing_replace].replace(missing_codes, np.nan)

In [ ]:
# group features
numeric_features = ['number_of_vehicles']

speed_limit_feature = ['speed_limit']

nominal_mode_impute = ['junction_detail', 'pedestrian_crossing', 'weather_conditions',
                        'carriageway_hazards', 'road_type', 'road_surface_conditions',
                        'light_conditions']

nominal_no_missing = ['month', 'time_of_day', 'first_road_class', 'urban_or_rural_area', 'day_of_week']

In [ ]:
# define transformers for each group of features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

speed_limit_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler())
])

nominal_mode_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

nominal_no_impute_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

In [ ]:
# combine all transformers
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('speed_limit', speed_limit_transformer, speed_limit_feature),
    ('nom_mode', nominal_mode_transformer, nominal_mode_impute),
    ('nom_no_impute', nominal_no_impute_transformer, nominal_no_missing),
], remainder='drop')

In [ ]:
# check the pipeline works
X_transformed_check = preprocessor.fit_transform(X, y)
print("Transformed shape:", X_transformed_check.shape)

Transformed shape: (100927, 72)


In [ ]:
# check for nan values after transformed
print(np.isnan(X_transformed_check.toarray()).sum())

0


In [ ]:
feature_names = preprocessor.get_feature_names_out()
X_transformed_df = pd.DataFrame(
    X_transformed_check.toarray() if hasattr(X_transformed_check, 'toarray') else X_transformed_check,
    columns=feature_names
)
X_transformed_df.head()

,num__number_of_vehicles,speed_limit__speed_limit,nom_mode__junction_detail_13.0,nom_mode__junction_detail_16.0,nom_mode__junction_detail_17.0,nom_mode__junction_detail_18.0,nom_mode__junction_detail_19.0,nom_mode__pedestrian_crossing_11.0,nom_mode__pedestrian_crossing_12.0,nom_mode__pedestrian_crossing_13.0,...,nom_no_impute__first_road_class_5,nom_no_impute__first_road_class_6,nom_no_impute__urban_or_rural_area_2,nom_no_impute__urban_or_rural_area_3,nom_no_impute__day_of_week_2,nom_no_impute__day_of_week_3,nom_no_impute__day_of_week_4,nom_no_impute__day_of_week_5,nom_no_impute__day_of_week_6,nom_no_impute__day_of_week_7
0,0.262929,-0.404584,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,-1.183999,-0.404584,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.262929,-0.404584,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,-1.183999,0.971416,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.262929,0.971416,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
